## Generating Normalized JSONs

In [1]:
import os

# --- import your functions ---
from src.ingestion_and_preprocessing import (
    partitioner,
    clean_json_elements,
    convert_tables_to_markdown,
    normalize_json_elements
)

# --- folder containing raw documents ---
INPUT_FOLDER = "dataset"

# --- process all files ---
for file_name in os.listdir(INPUT_FOLDER):
    file_path = os.path.join(INPUT_FOLDER, file_name)

    # skip non-files
    if not os.path.isfile(file_path):
        continue

    # --- only supported formats ---
    if not file_name.lower().endswith((".pdf", ".doc", ".docx")):
        continue

    print(f"\nProcessing: {file_name}")

    try:
        # 1. Partition → saves to parsed/
        partitioner(file_path)

        # --- construct parsed file path ---
        name, ext = os.path.splitext(file_name)
        ext = ext.lower()

        if ext == ".pdf":
            parsed_path = f"parsed/{name}_pdf.json"
        elif ext == ".doc":
            parsed_path = f"parsed/{name}_doc.json"
        elif ext == ".docx":
            parsed_path = f"parsed/{name}_docx.json"

        # 2a. Clean (in-place)
        clean_json_elements(parsed_path)

        # 2b. Table → Markdown (in-place)
        convert_tables_to_markdown(parsed_path)

        # 3. Normalize → saves to normalized/
        normalize_json_elements(parsed_path)

    except Exception as e:
        print(f"Error processing {file_name}: {e}")

c:\Users\SHREY\anaconda3\envs\uc_rag\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Processing: Computer Organization & Architecture_CSE2709.pdf


Loading weights: 100%|██████████| 367/367 [00:00<00:00, 6162.12it/s]


Saved parsed output to: parsed/Computer Organization & Architecture_CSE2709_pdf.json

File: parsed/Computer Organization & Architecture_CSE2709_pdf.json
Elements processed: 41
Removed (empty/invalid text): 0
Remaining elements: 41

File: parsed/Computer Organization & Architecture_CSE2709_pdf.json
Tables converted to markdown: 6

File processed: parsed/Computer Organization & Architecture_CSE2709_pdf.json
Elements normalized: 41
Skipped (invalid): 0
Saved to: normalized\Computer Organization & Architecture_CSE2709.json

Processing: Cryptography_CSE3703.pdf
Saved parsed output to: parsed/Cryptography_CSE3703_pdf.json

File: parsed/Cryptography_CSE3703_pdf.json
Elements processed: 42
Removed (empty/invalid text): 0
Remaining elements: 42

File: parsed/Cryptography_CSE3703_pdf.json
Tables converted to markdown: 7

File processed: parsed/Cryptography_CSE3703_pdf.json
Elements normalized: 42
Skipped (invalid): 0
Saved to: normalized\Cryptography_CSE3703.json

Processing: Deep Learning_CSE40

## Structural Chunking Preparation

In [2]:
import os
import json
from collections import Counter

# 🔧 Set your normalized folder path

folder_path = "normalized"  # change if needed

type_counter = Counter()
file_count = 0
element_count = 0

for filename in os.listdir(folder_path):
    if filename.endswith(".json"):
        file_path = os.path.join(folder_path, filename)
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)
            
            file_count += 1
            
            for el in data:
                element_count += 1
                el_type = el.get("type", "UNKNOWN")
                type_counter[el_type] += 1
# 📊 Print results

print(f"Processed {file_count} files")
print(f"Total elements: {element_count}\n")

print("Unique types and counts:\n")
for t, count in type_counter.most_common():
    print(f"{t}: {count}")

Processed 18 files
Total elements: 1119

Unique types and counts:

ListItem: 367
NarrativeText: 274
UncategorizedText: 180
Title: 175
markdown_text: 105
Image: 16
FigureCaption: 2


In [6]:
import os
import json
from collections import Counter, defaultdict

FOLDER_PATH = "normalized"  # change if needed

type_counter = Counter()
word_counts = defaultdict(int)

file_count = 0
element_count = 0

for filename in os.listdir(FOLDER_PATH):
    if filename.endswith(".json"):
        file_path = os.path.join(FOLDER_PATH, filename)
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)
            
            file_count += 1
            element_count += len(data)
            
            for el in data:
                el_type = el.get("type", "UNKNOWN")
                text = el.get("text", "")
                
                # Count words safely
                word_count = len(text.split()) if text else 0
                
                type_counter[el_type] += 1
                word_counts[el_type] += word_count

# --- Results ---

print(f"Processed files: {file_count}")
print(f"Total elements: {element_count}\n")

print(f"{'Type':<20} | {'Count':<6} | {'Avg Words':<10}")
print("-" * 42)

for t, count in type_counter.most_common():
    total_words = word_counts[t]
    avg_words = total_words / count if count > 0 else 0
    print(f"{t:<20} | {count:<6} | {avg_words:<10.2f}")

Processed files: 18
Total elements: 1119

Type                 | Count  | Avg Words 
------------------------------------------
ListItem             | 367    | 11.80     
NarrativeText        | 274    | 32.74     
UncategorizedText    | 180    | 3.19      
Title                | 175    | 3.55      
markdown_text        | 105    | 266.64    
Image                | 16     | 5.31      
FigureCaption        | 2      | 9.00      


## Chunking MiniLM

In [1]:
print("hello")

hello


In [2]:
from src.chunking import chunking_minilm_l6_v2

chunking_minilm_l6_v2(
    input_folder="normalized",
    chunk_size=256,
    overlap=26
)

c:\Users\SHREY\anaconda3\envs\uc_rag\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6952.59it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Chunking files: 100%|██████████| 18/18 [00:00<00:00, 32.04it/s]


✅ Chunking complete. Saved to: chunks/minilm_l6_v2_256_26
